# Diversity, Cold Start & Exploration

Companion notebook for the [Diversity, Cold Start & Exploration lesson](https://ml-viz-ruby.vercel.app/courses/recommender-systems/05-diversity-cold-start-exploration).

**The idea in one sentence.** A recommender that only maximises predicted
relevance produces a boring, redundant list, can't handle brand-new items, and
never learns about anything it doesn't already show — so production systems add
**diversity**, **cold-start**, and **exploration** on top of raw relevance.

The three tools, from scratch:

- **Maximal Marginal Relevance (MMR)** — greedily pick items that are relevant
  *and* dissimilar to what's already chosen, trading relevance for diversity.
- **Content-based cold start** — project a new item's features into the shared
  embedding space so it can be recommended before anyone clicks it.
- **Thompson Sampling** — explore by sampling from a Bayesian posterior, letting
  uncertain items get a chance to prove themselves.

We **validate that MMR raises intra-list diversity and Thompson concentrates on
the best arm**, then cover gotchas. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(0)

## 1 — Maximal Marginal Relevance (MMR)

MMR greedily selects items that are both relevant and diverse. At each step it balances the relevance score with similarity to already-selected items.

In [ ]:
def mmr(scores, emb, k, lam=0.5):
    """
    scores: (N,) relevance scores
    emb:    (N, d) item embeddings
    k:      number of items to select
    lam:    trade-off (1 = pure relevance, 0 = pure diversity)
    Returns indices of k selected items.
    """
    N = len(scores)
    selected, remaining = [], list(range(N))
    for _ in range(k):
        if not selected:
            # First item: pick highest relevance
            best = max(remaining, key=lambda i: scores[i])
        else:
            S_emb = emb[selected]                       # (|S|, d)
            def mmr_score(i):
                rel = scores[i]
                sim = float((emb[i] @ S_emb.T).max())  # max similarity to selected
                return lam * rel - (1 - lam) * sim
            best = max(remaining, key=mmr_score)
        selected.append(best)
        remaining.remove(best)
    return selected

# Create synthetic items: 50 items with d=8 embeddings and relevance scores
n, d = 50, 8
item_emb = rng.normal(size=(n, d))
item_emb /= np.linalg.norm(item_emb, axis=1, keepdims=True)
relevance = rng.uniform(0, 1, n)  # simulated relevance scores

top5_relevant = np.argsort(-relevance)[:5].tolist()
top5_diverse = mmr(relevance, item_emb, k=5, lam=0.5)

print("Top-5 by relevance only:", top5_relevant)
print("Top-5 by MMR (λ=0.5):  ", top5_diverse)

### Validate: MMR trades relevance for diversity

MMR's whole purpose is a *more diverse* top-k. We measure **intra-list distance**
(mean pairwise 1−cosine) for a relevance-only list vs an MMR list, and confirm MMR
is more diverse while a higher λ (more relevance weight) reduces that diversity.

In [ ]:
def ild(idxs):
    E = item_emb[idxs]; k = len(E); s = E @ E.T
    return sum(1 - s[i, j] for i in range(k) for j in range(k) if i != j) / (k * (k - 1))

top5_relevant = list(np.argsort(-relevance)[:5])
top5_mmr      = mmr(relevance, item_emb, k=5, lam=0.5)
top5_mmr_rel  = mmr(relevance, item_emb, k=5, lam=0.9)   # mostly relevance
print(f'ILD relevance-only : {ild(top5_relevant):.4f}')
print(f'ILD MMR (lam=0.5)  : {ild(top5_mmr):.4f}')
print(f'ILD MMR (lam=0.9)  : {ild(top5_mmr_rel):.4f}')
assert ild(top5_mmr) > ild(top5_relevant), 'MMR should be more diverse than relevance-only'
assert ild(top5_mmr) >= ild(top5_mmr_rel), 'lower lambda (more diversity weight) => more diverse'
print('\n✅ MMR raises intra-list diversity; lambda tunes the relevance/diversity trade-off')

## 2 — Content-based cold start

For a new item with no interactions, we project its content features into the item embedding space using a learned projection.

In [ ]:
# Simulate a content-feature -> embedding projection (pre-trained)
d_content, d_emb = 16, 8
W_proj = rng.normal(size=(d_emb, d_content)) * 0.1    # projection matrix

def cold_start_embedding(content_features):
    """Map content features to the shared embedding space."""
    emb = W_proj @ content_features
    return emb / np.linalg.norm(emb)

# New item: content features (e.g., text/image features)
new_item_content = rng.normal(size=d_content)
new_item_emb = cold_start_embedding(new_item_content)

# Find most similar existing items by cosine similarity
sims = item_emb @ new_item_emb
top3_similar = np.argsort(-sims)[:3]
print("New item embedding:", new_item_emb.round(3))
print("Most similar existing items:", top3_similar.tolist())
print("Similarity scores:", sims[top3_similar].round(3))

## 3 — Thompson Sampling

Thompson Sampling maintains a Beta posterior over each arm's click probability and samples from it to decide which item to recommend.

In [ ]:
class ThompsonSamplingBandit:
    def __init__(self, n_arms):
        self.alpha = np.ones(n_arms)   # successes + 1
        self.beta  = np.ones(n_arms)   # failures + 1

    def select_arm(self, rng_):
        samples = rng_.beta(self.alpha, self.beta)
        return int(np.argmax(samples))

    def update(self, arm, reward):
        self.alpha[arm] += reward
        self.beta[arm]  += (1 - reward)

# Simulate 200 rounds: 5 items with true click rates
n_arms = 5
true_rates = np.array([0.05, 0.15, 0.40, 0.20, 0.10])  # item 2 is best
bandit = ThompsonSamplingBandit(n_arms)

arm_counts = np.zeros(n_arms, dtype=int)
for t in range(200):
    arm = bandit.select_arm(rng)
    reward = float(rng.random() < true_rates[arm])
    bandit.update(arm, reward)
    arm_counts[arm] += 1

print("Arms pulled:", arm_counts)
print("Best arm (true):", true_rates.argmax(), f"(rate={true_rates.max():.2f})")
print("Most pulled arm:", arm_counts.argmax())

### Validate: Thompson Sampling concentrates on the best arm

A good bandit should pull the highest-click-rate item most often *without* being
told which it is. We check the most-pulled arm is the true best.

In [ ]:
assert arm_counts.argmax() == true_rates.argmax(), 'Thompson should pull the best arm most'
best = true_rates.argmax()
print(f'true best arm: {best} (rate {true_rates[best]:.2f})')
print(f'most-pulled arm: {arm_counts.argmax()} with {arm_counts.max()} of {arm_counts.sum()} pulls')
frac = arm_counts[best] / arm_counts.sum()
print(f'fraction of pulls on the best arm: {frac:.0%}')
assert frac > 0.3, 'the best arm should dominate the pulls'
print('\n✅ Thompson Sampling learned to exploit the best arm from click feedback alone')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **pure relevance → filter bubble** | redundant, boring lists; MMR (or DPP) injects diversity |
| **λ tuning** | too much diversity surfaces irrelevant items; too little is redundant |
| **cold-start content quality** | garbage features → garbage embedding; the projection is only as good as its inputs |
| **exploration hurts short-term metrics** | showing uncertain items costs immediate clicks but pays off long-term |
| **Thompson needs the right likelihood** | Beta posterior assumes Bernoulli clicks; other rewards need other priors |

Demo: content-based cold start — a new item's projected embedding lands near
content-similar items, so it's recommendable at launch.

In [ ]:
# Cold start really works: the projected embedding of a brand-new item lands near
# content-similar existing items, so it can be retrieved before it has ANY clicks.
new_c = rng.normal(size=d_content)
new_e = cold_start_embedding(new_c)
sims = item_emb @ new_e
# a near-duplicate content vector should map to a nearby embedding
dup_e = cold_start_embedding(new_c + 0.01 * rng.normal(size=d_content))
print(f'cosine(new item, near-duplicate content item): {new_e @ dup_e:.3f}')
assert new_e @ dup_e > 0.9, 'similar content must map to similar embeddings (that is what enables cold start)'
print('Content-similar items get nearby embeddings -> a new item is recommendable at launch.')

## ✏️ Your turn

**Exercise.** Implement `compute_ild(selected_emb)` — the Intra-List Diversity (ILD) metric: the average pairwise *cosine distance* (1 − cosine similarity) between all pairs of selected items.

Compare the ILD of the relevance-only top-5 selection vs. the MMR top-5 selection.

In [ ]:
def compute_ild(selected_emb):
    """
    selected_emb: (k, d) embeddings of selected items
    Returns: mean pairwise cosine distance (scalar)
    """
    # TODO(you): compute mean pairwise (1 - cosine_similarity) for all pairs i ≠ j
    return ...

ild_relevant = compute_ild(item_emb[top5_relevant])
ild_diverse  = compute_ild(item_emb[top5_diverse])
print(f"ILD (relevance-only): {ild_relevant:.4f}")
print(f"ILD (MMR λ=0.5):      {ild_diverse:.4f}")
print("MMR should be more diverse (higher ILD):", ild_diverse > ild_relevant)

In [ ]:
# Assertion cell
def _ref_ild(E):
    k = len(E)
    sims = E @ E.T
    total = sum(1 - sims[i,j] for i in range(k) for j in range(k) if i != j)
    return total / (k*(k-1))
assert abs(compute_ild(item_emb[top5_relevant]) - _ref_ild(item_emb[top5_relevant])) < 1e-6
print("✓ ILD implementation is correct")

<details>
<summary>Solution</summary>

```python
def compute_ild(selected_emb):
    k = len(selected_emb)
    sims = selected_emb @ selected_emb.T   # (k, k) cosine similarities
    distances = 1 - sims
    # sum all off-diagonal entries, divide by k*(k-1)
    return (distances.sum() - np.trace(distances)) / (k * (k - 1))
```
</details>

## Key takeaways

- **Relevance alone is not enough.** Production recsys balances it against
  **diversity** (avoid redundancy), **cold start** (score new items), and
  **exploration** (keep learning).
- **MMR** greedily trades relevance for diversity via λ — we measured the rise in
  intra-list distance.
- **Content-based cold start** projects item features into the shared space, so a
  new item is recommendable before its first click.
- **Thompson Sampling** explores by posterior sampling and concentrates on the
  best arm from feedback alone — no hand-tuned ε needed.